In [ ]:
from sklearn.model_selection import train_test_split
import os
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F

In [ ]:


data_dir = r"C:\Users\Aman\shakespeare\data"

rows = []

for file in os.listdir(data_dir):
    if file.endswith("_original.snt.aligned"):

        play = file.replace("_original.snt.aligned", "")
        original_path = os.path.join(data_dir, file)
        modern_path = os.path.join(
            data_dir,
            play + "_modern.snt.aligned"
        )

        if not os.path.exists(modern_path):
            continue

        with open(original_path, "r", encoding="utf-8") as f:
            original = f.readlines()

        with open(modern_path, "r", encoding="utf-8") as f:
            modern = f.readlines()

        for modern_text, shakespeare_text in zip(modern, original):

            modern_text = modern_text.strip()
            shakespeare_text = shakespeare_text.strip()

            if modern_text and shakespeare_text:

                rows.append({
                    "play": play,
                    "modern": modern_text,
                    "shakespeare": shakespeare_text
                })


df = pd.DataFrame(rows)

print(df.shape)
print(df.head())
print(df["play"].value_counts())

In [ ]:
df.head()

In [ ]:
from tokenizers import ByteLevelBPETokenizer
from transformers import PreTrainedTokenizerFast

bpe = ByteLevelBPETokenizer()
bpe.train_from_iterator(
    df["modern"].tolist() + df["shakespeare"].tolist(),
    vocab_size=8000,
    special_tokens=["<PAD>", "<EOS>"]
)
bpe.save("shakespeare_bpe.json")   # save as single json, not save_model

tokenizer = PreTrainedTokenizerFast(tokenizer_file="shakespeare_bpe.json")
tokenizer.pad_token = "<PAD>"
tokenizer.eos_token = "<EOS>"

In [ ]:
l=df['play'].unique()

length=[]
for i in l:
    length.append(len(df[df['play']==i]))
average_length=sum(length)/len(length)
average_length

    

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super(DecoderBlock, self).__init__()
        self.n_heads = n_heads
        self.head_size=d_model // n_heads
        self.query = nn.Linear(d_model,d_model,bias=False)
        self.key = nn.Linear(d_model,d_model,bias=False)
        self.value = nn.Linear(d_model,d_model,bias=False)
        self.cross_query = nn.Linear(d_model, d_model, bias=False)
        self.cross_key = nn.Linear(d_model, d_model, bias=False)
        self.cross_value = nn.Linear(d_model, d_model, bias=False)
        self.cross_proj = nn.Linear(d_model, d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.ln3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
    def masked_self_attention(self,x, mask=None):
        B,T,C=x.shape
        k=self.key(x)
        q=self.query(x)
        v=self.value(x)
        q = q.view(B, T, self.n_heads, self.head_size)
        k = k.view(B, T, self.n_heads, self.head_size)
        v = v.view(B, T, self.n_heads, self.head_size)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        wei=q@k.transpose(-2,-1) / (self.head_size ** 0.5)
        tril=torch.tril(torch.ones(T,T,device=x.device)).bool()
        causal = tril.view(1,1,T,T)
        keep = (causal & mask.bool()) if mask is not None else causal
        wei=wei.masked_fill(~keep,float('-inf'))
        wei=F.softmax(wei,dim=-1)
        out=wei@v
        out = out.transpose(1, 2)
        out = out.contiguous().view(B, T, C)
        out = self.proj(out)
     
        return out
    def cross_attention(self, x, encoder_output, encoder_mask=None):
        B, T, C = x.shape
        S = encoder_output.shape[1]
        k=self.cross_key(encoder_output)
        q=self.cross_query(x)
        v=self.cross_value(encoder_output)
        q = q.view(B, T, self.n_heads, self.head_size)
        k = k.view(B, S, self.n_heads, self.head_size)
        v = v.view(B, S, self.n_heads, self.head_size)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        wei=q@k.transpose(-2,-1) / (self.head_size ** 0.5)
        if encoder_mask is not None:
            wei = wei.masked_fill(encoder_mask == 0, float('-inf'))
        wei=F.softmax(wei,dim=-1)
        out=wei@v
        out = out.transpose(1, 2)
        out = out.contiguous().view(B, T, C)
        out = self.cross_proj(out)
        return out
    def forward(self, x, encoder_output, self_mask=None, cross_mask=None):

        x = x + self.dropout(
            self.masked_self_attention(self.ln1(x),self_mask)
        )

        x = x + self.dropout(
            self.cross_attention(
                self.ln2(x),
                encoder_output,
                cross_mask
            )
        )

        x = x + self.ffn(self.ln3(x))

        return x

In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=42, shuffle=True
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, shuffle=True
)
train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

In [ ]:
text = "Wherefore art thou"

tokens = tokenizer.encode(text)

print(tokens)

In [ ]:
tokenizer.add_special_tokens({
    "pad_token": "<PAD>"
})

print("EOS:", tokenizer.eos_token_id)
print("PAD:", tokenizer.pad_token_id)
print("Vocab size:", len(tokenizer))
tokenizer.pad_token_id

In [ ]:
sample = train_df.iloc[0]

print(sample["modern"])
print(sample["shakespeare"])
modern_ids = tokenizer.encode(
    sample["modern"],
    truncation=True,
    max_length=256
)

shakespeare_ids = tokenizer.encode(
    sample["shakespeare"],
    truncation=True,
    max_length=256
)

print(modern_ids)
print(shakespeare_ids)

In [ ]:
MAX_LENGTH = 256

def encode_with_eos(texts, tokenizer, max_length):

    input_ids_batch = []
    attn_batch = []
    for t in texts:
        ids = tokenizer.encode(t, truncation=True, max_length=max_length - 1)
        ids = ids + [tokenizer.eos_token_id]
        ids = ids[:max_length]
        pad_len = max_length - len(ids)
        attn = [1] * len(ids) + [0] * pad_len
        ids = ids + [tokenizer.pad_token_id] * pad_len
        input_ids_batch.append(ids)
        attn_batch.append(attn)
    return {"input_ids": input_ids_batch, "attention_mask": attn_batch}

# encoder side (modern) — no EOS needed, plain padding is fine
train_inputs = tokenizer(
    train_df["modern"].tolist(),
    padding="max_length", truncation=True, max_length=MAX_LENGTH
)
val_inputs = tokenizer(
    val_df["modern"].tolist(),
    padding="max_length", truncation=True, max_length=MAX_LENGTH
)
test_inputs = tokenizer(
    test_df["modern"].tolist(),
    padding="max_length", truncation=True, max_length=MAX_LENGTH
)

train_targets = encode_with_eos(train_df["shakespeare"].tolist(), tokenizer, MAX_LENGTH)
val_targets   = encode_with_eos(val_df["shakespeare"].tolist(), tokenizer, MAX_LENGTH)
test_targets  = encode_with_eos(test_df["shakespeare"].tolist(), tokenizer, MAX_LENGTH)

In [35]:
import torch
from torch.utils.data import Dataset

class ShakespeareDataset(Dataset):

    def __init__(self, inputs, targets):
        self.input_ids = inputs["input_ids"]
        self.input_mask = inputs["attention_mask"] 
        self.target_ids = targets["input_ids"]
        self.target_mask = targets["attention_mask"] 

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):

        input_ids = torch.tensor(
            self.input_ids[idx],
            dtype=torch.long
        )
        input_mask = torch.tensor(self.input_mask[idx], dtype=torch.long)

        target_ids = torch.tensor(
            self.target_ids[idx],
            dtype=torch.long
        )
        target_mask = torch.tensor(self.target_mask[idx], dtype=torch.long)
        decoder_mask = target_mask.clone()         
        decoder_mask[1:] = target_mask[:-1]
        decoder_mask[0] = 1
        decoder_input = target_ids.clone()
        decoder_input[1:] = target_ids[:-1]
        decoder_input[0] = tokenizer.eos_token_id

        labels = target_ids

        return {
            "input_ids": input_ids,
            "input_mask": input_mask,
            "decoder_input": decoder_input,
            "decoder_mask": decoder_mask,
            "labels": labels
        }

In [36]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_dataset = ShakespeareDataset(train_inputs, train_targets)
val_dataset = ShakespeareDataset(val_inputs, val_targets)
test_dataset = ShakespeareDataset(test_inputs, test_targets)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [37]:
#i need to tune this
d_model = 384
n_heads = 6
num_layers = 4
d_ff = 1536
dropout = 0.1
max_length = 256
vocab_size = len(tokenizer)

In [38]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.25):
        super(EncoderBlock, self).__init__()
        self.n_heads = n_heads
        self.head_size=d_model // n_heads
        self.query = nn.Linear(d_model,d_model,bias=False)
        self.key = nn.Linear(d_model,d_model,bias=False)
        self.value = nn.Linear(d_model,d_model,bias=False)
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
    def attention(self,x, mask=None):
        B,T,C=x.shape
        k=self.key(x)
        q=self.query(x)
        v=self.value(x)
        q = q.view(B, T, self.n_heads, self.head_size)
        k = k.view(B, T, self.n_heads, self.head_size)
        v = v.view(B, T, self.n_heads, self.head_size)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        wei=q@k.transpose(-2,-1) / (self.head_size ** 0.5)
        if mask is not None:
            wei = wei.masked_fill(mask == 0, float('-inf'))
        wei=F.softmax(wei,dim=-1)
        out=wei@v
        out = out.transpose(1, 2)
        out = out.contiguous().view(B, T, C)
        out = self.proj(out)
     
        return out
    def forward(self, x, mask=None):
        x = x + self.dropout(self.attention(self.ln1(x),mask))
        x = x + self.ffn(self.ln2(x))
        return x

In [39]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super(DecoderBlock, self).__init__()
        self.n_heads = n_heads
        self.head_size=d_model // n_heads
        self.query = nn.Linear(d_model,d_model,bias=False)
        self.key = nn.Linear(d_model,d_model,bias=False)
        self.value = nn.Linear(d_model,d_model,bias=False)
        self.cross_query = nn.Linear(d_model, d_model, bias=False)
        self.cross_key = nn.Linear(d_model, d_model, bias=False)
        self.cross_value = nn.Linear(d_model, d_model, bias=False)
        self.cross_proj = nn.Linear(d_model, d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.ln3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
    def masked_self_attention(self,x, mask=None):
        B,T,C=x.shape
        k=self.key(x)
        q=self.query(x)
        v=self.value(x)
        q = q.view(B, T, self.n_heads, self.head_size)
        k = k.view(B, T, self.n_heads, self.head_size)
        v = v.view(B, T, self.n_heads, self.head_size)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        wei=q@k.transpose(-2,-1) / (self.head_size ** 0.5)
        tril=torch.tril(torch.ones(T,T,device=x.device)).bool()
        causal = tril.view(1,1,T,T)
        keep = (causal & mask.bool()) if mask is not None else causal
        wei=wei.masked_fill(~keep,float('-inf'))
        wei=F.softmax(wei,dim=-1)
        out=wei@v
        out = out.transpose(1, 2)
        out = out.contiguous().view(B, T, C)
        out = self.proj(out)
     
        return out
    def cross_attention(self, x, encoder_output, encoder_mask=None):
        B, T, C = x.shape
        S = encoder_output.shape[1]
        k=self.cross_key(encoder_output)
        q=self.cross_query(x)
        v=self.cross_value(encoder_output)
        q = q.view(B, T, self.n_heads, self.head_size)
        k = k.view(B, S, self.n_heads, self.head_size)
        v = v.view(B, S, self.n_heads, self.head_size)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        wei=q@k.transpose(-2,-1) / (self.head_size ** 0.5)
        if encoder_mask is not None:
            wei = wei.masked_fill(encoder_mask == 0, float('-inf'))
        wei=F.softmax(wei,dim=-1)
        out=wei@v
        out = out.transpose(1, 2)
        out = out.contiguous().view(B, T, C)
        out = self.cross_proj(out)
        return out
    def forward(self, x, encoder_output, self_mask=None, cross_mask=None):

        x = x + self.dropout(
            self.masked_self_attention(self.ln1(x),self_mask)
        )

        x = x + self.dropout(
            self.cross_attention(
                self.ln2(x),
                encoder_output,
                cross_mask
            )
        )

        x = x + self.ffn(self.ln3(x))

        return x

In [40]:

class TransformerModel(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, num_layers, d_ff, dropout, max_length):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_length, d_model)
        
        self.blocks = nn.ModuleList([
            EncoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.decoder_blocks = nn.ModuleList([
            DecoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.lm_head = nn.Linear(d_model, vocab_size)
        self.lm_head.weight = self.token_embedding.weight
        
    def forward(self, x, target, src_mask=None, tgt_mask=None):
        B,T=x.shape
        token_emb =self.token_embedding(x)
        positions=torch.arange(T,device=x.device)
        target_emb = self.token_embedding(target)
        target_positions = torch.arange(target.shape[1],device=target.device)
        target_pos_emb = self.position_embedding(target_positions)
        target = target_emb + target_pos_emb
        
        pos_emb = self.position_embedding(positions)

        x = token_emb + pos_emb
        enc_mask = src_mask[:, None, None, :] if src_mask is not None else None
        dec_mask = tgt_mask[:, None, None, :] if tgt_mask is not None else None
        for block in self.blocks:
                x = block(x, enc_mask)
        encoder_output = x
        for block in self.decoder_blocks:
            target = block(target, encoder_output, dec_mask, enc_mask)
        
        

        logits = self.lm_head(target)
        return logits
    

In [41]:
lr = 0.001
beta1 = 0.9
beta2 = 0.999
eps = 1e-8
class Adam:
    def __init__(self, parameters, lr=0.001,
                 beta1=0.9, beta2=0.999, eps=1e-8):

        self.parameters = list(parameters)

        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps

        self.m = []
        self.v = []

        for param in self.parameters:
            self.m.append(torch.zeros_like(param))
            self.v.append(torch.zeros_like(param))

        self.t = 0
    def step(self):
        self.t += 1
        for i, param in enumerate(self.parameters):
            if param.grad is None:
                continue

            grad = param.grad.data

            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * grad
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * (grad ** 2)

            m_hat = self.m[i] / (1 - self.beta1 ** self.t)
            v_hat = self.v[i] / (1 - self.beta2 ** self.t)

            update = self.lr * m_hat / (torch.sqrt(v_hat) + self.eps)

            param.data -= update

In [42]:
model = TransformerModel(
    vocab_size=vocab_size,
    d_model=d_model,
    n_heads=n_heads,
    num_layers=num_layers,
    d_ff=d_ff,
    dropout=dropout,
    max_length=max_length
)

In [43]:
batch = next(iter(train_loader))

input_ids = batch["input_ids"]
decoder_input = batch["decoder_input"]
labels = batch["labels"]

In [44]:
input_ids = batch["input_ids"]
input_mask = batch["input_mask"]
decoder_input = batch["decoder_input"]
decoder_mask = batch["decoder_mask"]
labels = batch["labels"]

logits = model(input_ids, decoder_input, input_mask, decoder_mask)

In [45]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

print(device)

cuda


In [46]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=0.05
)

In [47]:
loss_fn = nn.CrossEntropyLoss(
    ignore_index=tokenizer.pad_token_id,
    label_smoothing=0.1
)

In [48]:
epochs = 15

CHECKPOINT_PATH = "model_checkpoint.pth"
saved_epoch = 0

if os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(CHECKPOINT_PATH)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    saved_epoch = checkpoint["epoch"]
    print(f"Resuming from epoch {saved_epoch}")
else:
    print("No checkpoint found, training from scratch")

for epoch in range(saved_epoch, epochs):
    
    
    model.train()

    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        input_mask = batch["input_mask"].to(device)
        decoder_input = batch["decoder_input"].to(device)
        decoder_mask = batch["decoder_mask"].to(device)
        labels = batch["labels"].to(device)
        optimizer.zero_grad()

        logits = model(input_ids, decoder_input, input_mask, decoder_mask)

        loss = loss_fn(
            logits.reshape(-1, vocab_size),
            labels.reshape(-1)
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1}/{epochs} "
        f"Loss: {avg_loss:.4f}"
    )
    torch.save({
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss": avg_loss
    }, CHECKPOINT_PATH)


No checkpoint found, training from scratch
Epoch 1/15 Loss: 33.6640
Epoch 2/15 Loss: 8.0886
Epoch 3/15 Loss: 6.8858
Epoch 4/15 Loss: 6.3027
Epoch 5/15 Loss: 5.9493
Epoch 6/15 Loss: 5.6832
Epoch 7/15 Loss: 5.4843
Epoch 8/15 Loss: 5.2984


KeyboardInterrupt: 

In [49]:
checkpoint = torch.load(CHECKPOINT_PATH)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

val_loss = 0

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        input_mask = batch["input_mask"].to(device)
        decoder_input = batch["decoder_input"].to(device)
        decoder_mask = batch["decoder_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids, decoder_input, input_mask, decoder_mask)

        loss = loss_fn(
            logits.reshape(-1, vocab_size),
            labels.reshape(-1)
        )
        val_loss += loss.item()

val_loss /= len(val_loader)

print(f"Val Loss: {val_loss:.4f}")

Val Loss: 5.4971


In [50]:
def translate(sentence, max_length=64, do_sample=False,
              temperature=0.8, top_k=40, top_p=0.9,
              repetition_penalty=1.3, no_repeat_ngram=3):

    model.eval()

    source = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )
    input_ids = source["input_ids"].to(device)

    generated = torch.tensor(
        [[tokenizer.eos_token_id]],
        dtype=torch.long,
        device=device
    )

    with torch.no_grad():
        for _ in range(max_length):

            logits = model(input_ids, generated)
            next_logits = logits[0, -1, :].clone()

            # --- repetition penalty: downweight tokens already used ---
            for tok in set(generated[0].tolist()):
                if next_logits[tok] > 0:
                    next_logits[tok] /= repetition_penalty
                else:
                    next_logits[tok] *= repetition_penalty

            # --- block repeating the last (n-1)-gram exactly ---
            if generated.shape[1] >= no_repeat_ngram - 1:
                prefix = generated[0, -(no_repeat_ngram - 1):].tolist()
                for i in range(generated.shape[1] - (no_repeat_ngram - 1)):
                    if generated[0, i:i + no_repeat_ngram - 1].tolist() == prefix:
                        banned_tok = generated[0, i + no_repeat_ngram - 1].item()
                        next_logits[banned_tok] = float('-inf')

            if do_sample:
                next_logits = next_logits / temperature

                # top-k filtering
                if top_k > 0:
                    topk_vals, topk_idx = torch.topk(next_logits, top_k)
                    filtered = torch.full_like(next_logits, float('-inf'))
                    filtered[topk_idx] = topk_vals
                    next_logits = filtered

                # top-p (nucleus) filtering
                if top_p < 1.0:
                    sorted_logits, sorted_idx = torch.sort(next_logits, descending=True)
                    probs = F.softmax(sorted_logits, dim=-1)
                    cum_probs = torch.cumsum(probs, dim=-1)
                    remove = cum_probs > top_p
                    remove[1:] = remove[:-1].clone()
                    remove[0] = False
                    sorted_logits[remove] = float('-inf')
                    next_logits = torch.full_like(next_logits, float('-inf'))
                    next_logits[sorted_idx] = sorted_logits

                probs = F.softmax(next_logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1).unsqueeze(0)
            else:
                next_token = torch.argmax(next_logits, dim=-1, keepdim=True).unsqueeze(0)

            generated = torch.cat([generated, next_token], dim=1)

            if next_token.item() == tokenizer.eos_token_id:
                break

    result = tokenizer.decode(generated[0], skip_special_tokens=True)
    return result

In [51]:
sentence = "Where are you going?"
model.eval()
result = translate(sentence)

print("Modern:")
print(sentence)

print("\nShakespeare:")
print(result)

Modern:
Where are you going?

Shakespeare:
Whither go?


In [52]:
sentence = train_df.iloc[1]["modern"]

print("Modern:")
print(sentence)

print("\nActual:")
print(train_df.iloc[1]["shakespeare"])

print("\nModel:")
print(translate(sentence))

Modern:
I should always be wise.

Actual:
I should be wise, for honesty’s a fool And loses that it works for.

Model:
I should be wise, as I may.
